In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.utils import add_self_loops, softmax

import warnings
warnings.filterwarnings('ignore')

/opt/anaconda3/envs/topicgraph_py310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = Planetoid(root='./data', name='Cora')
data = dataset[0]

print(data)
print(dataset.num_features)
print(dataset.num_classes)
print(data.num_nodes)
print(data.num_edges)
print(data.train_mask.sum())
print(data.has_self_loops())
print(data.is_undirected())

Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])
1433
7
2708
10556
tensor(140)
False
True


In [3]:
class GATLayer(nn.Module):
    def __init__(self, in_channels, out_channels, dropout=0.6, alpha=0.2):
        super().__init__()

        self.W = nn.Linear(in_channels, out_channels, bias=False)
        self.a = nn.Parameter(torch.empty(2 * out_channels, 1))

        self.dropout = dropout
        self.alpha = alpha

        nn.init.xavier_uniform_(self.W.weight)
        nn.init.xavier_uniform_(self.a)

    def forward(self, x, edge_index, num_nodes, return_attention=False):
        x = F.dropout(x, p=self.dropout, training=self.training)

        edge_index, _ = add_self_loops(edge_index, num_nodes=num_nodes)
        row, col = edge_index

        x = self.W(x)

        x_i = x[col]
        x_j = x[row]

        e = torch.cat([x_i, x_j], dim=1) @ self.a
        e = F.leaky_relu(e.squeeze(-1), negative_slope=self.alpha)

        alpha = softmax(e, col)

        alpha = F.dropout(alpha, p=self.dropout, training=self.training)

        out = torch.zeros(num_nodes, x.size(1), device=x.device)
        out.index_add_(0, col, alpha.unsqueeze(-1) * x_j)

        if return_attention:
            return out, alpha, edge_index
        return out
    

class MultiHeadGATLayer(nn.Module):
    def __init__(self, in_channels, out_channels, heads, concat=True, dropout=0.6, alpha=0.2):
        super().__init__()

        self.concat = concat
        self.heads = nn.ModuleList([
            GATLayer(in_channels, out_channels, dropout=dropout, alpha=alpha)
            for _ in range(heads)
        ])
    
    def forward(self, x, edge_index, num_nodes):
        head_outs = [head(x, edge_index, num_nodes) for head in self.heads]
    
        if self.concat:
            return torch.cat(head_outs, dim=1)
        else:
            return torch.mean(torch.stack(head_outs, dim=0), dim=0)

In [4]:
class GAT(nn.Module):
    def __init__(
        self,
        in_channels,
        hidden_channels,
        out_channels,
        num_heads_first=8,
        num_heads_second=1,
        dropout=0.6,
        alpha=0.2
    ):
        super().__init__()

        self.dropout = dropout

        # 1st layer: 8 heads, each head outputs 8 features
        self.gat1 = MultiHeadGATLayer(
            in_channels=in_channels,
            out_channels=hidden_channels,
            heads=num_heads_first,
            concat=True,
            dropout=dropout,
            alpha=alpha
        )

        # 2nd layer: final prediction layer
        self.gat2 = MultiHeadGATLayer(
            in_channels=hidden_channels * num_heads_first,
            out_channels=out_channels,
            heads=num_heads_second,
            concat=False,
            dropout=dropout,
            alpha=alpha
        )

    def forward(self, x, edge_index, num_nodes):
        x = self.gat1(x, edge_index, num_nodes)
        x = F.elu(x)  # 논문 hidden layer activation
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.gat2(x, edge_index, num_nodes)
        return x

In [5]:
model = GAT(
    in_channels=dataset.num_features,
    hidden_channels=8,
    out_channels=dataset.num_classes,
    num_heads_first=8,
    num_heads_second=1,
    dropout=0.6,
    alpha=0.2
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.005,
    weight_decay=5e-4
)

criterion = nn.CrossEntropyLoss()

def train():
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index, data.num_nodes)
    loss = criterion(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()

def test():
    model.eval()
    out = model(data.x, data.edge_index, data.num_nodes)
    pred = out.argmax(dim=1)
    results = {}
    for split, mask in [('train', data.train_mask),
                        ('val',   data.val_mask),
                        ('test',  data.test_mask)]:
        correct = (pred[mask] == data.y[mask]).sum().item()
        results[split] = correct / mask.sum().item()
    return results

In [6]:
best_val_acc = 0
best_test_acc = 0

for epoch in range(1, 201):
    loss = train()
    accs = test()

    if accs['val'] > best_val_acc:
        best_val_acc = accs['val']
        best_test_acc = accs['test']

    if epoch % 20 == 0:
        print(f"Epoch {epoch:03d} | Loss: {loss:.4f} | "
              f"Train: {accs['train']:.4f} | "
              f"Val: {accs['val']:.4f} | "
              f"Test: {accs['test']:.4f}")

print(f"\n Best Val: {best_val_acc:.4f} | Best Test: {best_test_acc:.4f}")
print(f" Paper target: ~0.83")

Epoch 020 | Loss: 1.2210 | Train: 0.9286 | Val: 0.8060 | Test: 0.8030
Epoch 040 | Loss: 1.0020 | Train: 0.9857 | Val: 0.7980 | Test: 0.8230
Epoch 060 | Loss: 0.6734 | Train: 0.9929 | Val: 0.8000 | Test: 0.8090
Epoch 080 | Loss: 0.6943 | Train: 0.9929 | Val: 0.7960 | Test: 0.8030
Epoch 100 | Loss: 0.5574 | Train: 1.0000 | Val: 0.7960 | Test: 0.8030
Epoch 120 | Loss: 0.6137 | Train: 1.0000 | Val: 0.8020 | Test: 0.8010
Epoch 140 | Loss: 0.4826 | Train: 1.0000 | Val: 0.7900 | Test: 0.8020
Epoch 160 | Loss: 0.5746 | Train: 1.0000 | Val: 0.7940 | Test: 0.8030
Epoch 180 | Loss: 0.3488 | Train: 1.0000 | Val: 0.7840 | Test: 0.7950
Epoch 200 | Loss: 0.5542 | Train: 1.0000 | Val: 0.7780 | Test: 0.7950

 Best Val: 0.8060 | Best Test: 0.8030
 Paper target: ~0.83
